# Small RL-Style Policy Gradient on MNIST (Colab Friendly)

This notebook compares policy-gradient estimators on an image classification task (MNIST).

Rollout per image: sample `N` labels from the model categorical distribution.
Reward per rollout: `r=1` if sampled label equals ground truth else `0`.

Implemented estimators:
- `rl`: `1/N * sum(grad(log p) * r)`
- `maxrl`: `1/num_correct * sum(grad(log p) * (r-r_mean))`
- `grpo`: `1/N * sum(grad(log p) * (r-r_mean)/r_std)`
- `winner_only`: `1/num_correct * sum_correct(grad(log p) * (1-r_mean))`
- `raft`: `1/num_correct * sum_correct(grad(log p))`
- `ce`: standard supervised cross-entropy

If budget is tight, keep `FULL_COMPARE=False` (default) to compare only `maxrl` vs `winner_only`.

In [ ]:
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)

FULL_COMPARE = False  # set True to run all methods
METHODS = (
    ["ce", "maxrl", "winner_only"]
    if not FULL_COMPARE
    else ["ce", "rl", "maxrl", "grpo", "winner_only", "raft"]
)
N_VALUES = [2, 4, 8, 16] if not FULL_COMPARE else [2, 4, 8]
EPOCHS = 3
BATCH_SIZE = 128
LR = 3e-4
TRAIN_SUBSET = 6000
TEST_SUBSET = 2000

print("METHODS:", METHODS)
print("N_VALUES:", N_VALUES)

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_ds_full = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_ds_full = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_idx = np.random.RandomState(0).choice(
    len(train_ds_full), TRAIN_SUBSET, replace=False
)
test_idx = np.random.RandomState(1).choice(
    len(test_ds_full), TEST_SUBSET, replace=False
)

train_ds = Subset(train_ds_full, train_idx)
test_ds = Subset(test_ds_full, test_idx)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True
)

len(train_ds), len(test_ds)

In [ ]:
class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)


def evaluate_argmax_acc(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return correct / max(total, 1)

In [ ]:
def estimator_loss(logits, y, N, method):
    # logits: [B, C], y: [B]
    dist = Categorical(logits=logits)

    if method == "ce":
        ce_loss = nn.CrossEntropyLoss()(logits, y)
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
            sampled = probs.multinomial(num_samples=N, replacement=True).transpose(0, 1)
            rewards = (sampled == y.unsqueeze(0)).float()
            batch_reward_mean = rewards.mean().item()
        batch_correct_frac = batch_reward_mean
        return ce_loss, batch_reward_mean, batch_correct_frac

    samples = dist.sample((N,))  # [N, B]
    rewards = (samples == y.unsqueeze(0)).float()  # [N, B]
    logp = dist.log_prob(samples)  # [N, B]

    r_mean = rewards.mean(dim=0, keepdim=True)  # [1, B]
    r_std = rewards.std(dim=0, keepdim=True, unbiased=False).clamp_min(1e-6)
    raw_num_correct = rewards.sum(dim=0, keepdim=True)
    num_correct = raw_num_correct.clamp_min(1.0)

    if method == "rl":
        weights = rewards / float(N)
    elif method == "maxrl":
        weights = (rewards - r_mean) / num_correct
    elif method == "grpo":
        weights = ((rewards - r_mean) / r_std) / float(N)
    elif method == "winner_only":
        weights = (rewards * (1.0 - r_mean)) / num_correct
    elif method == "raft":
        weights = rewards / num_correct
    else:
        raise ValueError(f"Unknown method: {method}")

    # maximize expected weighted log-prob -> minimize negative
    loss = -(weights * logp).sum(dim=0).mean()

    batch_reward_mean = rewards.mean().item()
    batch_correct_frac = (raw_num_correct > 0).float().mean().item()
    return loss, batch_reward_mean, batch_correct_frac

In [ ]:
@dataclass
class TrainRecord:
    method: str
    N: int
    epoch: int
    train_reward: float
    train_has_winner: float
    test_acc: float


def train_one_setting(method, N, epochs=3):
    set_seed(0)
    model = SmallMLP().to(device)
    opt = optim.Adam(model.parameters(), lr=LR)

    records = []
    for ep in range(1, epochs + 1):
        model.train()
        reward_meter = []
        winner_meter = []

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss, batch_reward, batch_has_winner = estimator_loss(
                logits, y, N=N, method=method
            )

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            reward_meter.append(batch_reward)
            winner_meter.append(batch_has_winner)

        test_acc = evaluate_argmax_acc(model, test_loader)
        rec = TrainRecord(
            method=method,
            N=N,
            epoch=ep,
            train_reward=float(np.mean(reward_meter)),
            train_has_winner=float(np.mean(winner_meter)),
            test_acc=test_acc,
        )
        records.append(rec)
        print(
            f"[{method:11s}] N={N:2d} epoch={ep}: train_reward={rec.train_reward:.4f}, test_acc={rec.test_acc:.4f}"
        )

    return records

In [ ]:
all_records = []

# CE baseline does not depend on rollout count; run it once with N=1
if "ce" in METHODS:
    all_records.extend(train_one_setting(method="ce", N=1, epochs=EPOCHS))

for method in METHODS:
    if method == "ce":
        continue
    for N in N_VALUES:
        all_records.extend(train_one_setting(method=method, N=N, epochs=EPOCHS))

df = pd.DataFrame([r.__dict__ for r in all_records])
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

for (method, N), g in df.groupby(["method", "N"]):
    g = g.sort_values("epoch")
    label = f"{method}, N={N}" if method != "ce" else "ce baseline"
    style = (
        {"linestyle": "--", "linewidth": 2, "marker": "s"}
        if method == "ce"
        else {"marker": "o"}
    )
    axes[0].plot(g["epoch"], g["test_acc"], label=label, **style)
    axes[1].plot(g["epoch"], g["train_reward"], label=label, **style)

axes[0].set_title("Test Accuracy vs Epoch (All Methods)")
axes[0].set_ylabel("Argmax Accuracy")
axes[0].set_xlabel("Epoch")

axes[1].set_title("Train Rollout Reward Mean vs Epoch (All Methods)")
axes[1].set_ylabel("Mean Reward over sampled rollouts")
axes[1].set_xlabel("Epoch")

handles, labels = axes[0].get_legend_handles_labels()
# de-duplicate labels while preserving order
seen = set()
u_handles, u_labels = [], []
for handle, label in zip(handles, labels):
    if label in seen:
        continue
    seen.add(label)
    u_handles.append(handle)
    u_labels.append(label)
fig.legend(u_handles, u_labels, loc="center right", bbox_to_anchor=(1.18, 0.5))
plt.tight_layout()
plt.show()

In [ ]:
non_ce_methods = [m for m in METHODS if m != "ce"]
if len(non_ce_methods) == 0:
    print("No non-CE methods selected.")
else:
    ce_df = df[df["method"] == "ce"].sort_values("epoch")
    for method in non_ce_methods:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

        mdf = df[df["method"] == method]
        for N, g in mdf.groupby("N"):
            g = g.sort_values("epoch")
            axes[0].plot(
                g["epoch"], g["test_acc"], marker="o", label=f"{method}, N={N}"
            )
            axes[1].plot(
                g["epoch"], g["train_reward"], marker="o", label=f"{method}, N={N}"
            )

        if not ce_df.empty:
            axes[0].plot(
                ce_df["epoch"],
                ce_df["test_acc"],
                marker="s",
                linestyle="--",
                linewidth=2,
                label="ce baseline",
            )
            axes[1].plot(
                ce_df["epoch"],
                ce_df["train_reward"],
                marker="s",
                linestyle="--",
                linewidth=2,
                label="ce baseline",
            )

        axes[0].set_title(f"Test Accuracy: {method} vs CE")
        axes[0].set_ylabel("Argmax Accuracy")
        axes[0].set_xlabel("Epoch")

        axes[1].set_title(f"Train Rollout Reward: {method} vs CE")
        axes[1].set_ylabel("Mean Reward over sampled rollouts")
        axes[1].set_xlabel("Epoch")

        axes[0].legend()
        axes[1].legend()
        plt.tight_layout()
        plt.show()

In [ ]:
summary = (
    df.sort_values("epoch")
    .groupby(["method", "N"])
    .tail(1)
    .sort_values(["test_acc"], ascending=False)[
        ["method", "N", "test_acc", "train_reward", "train_has_winner"]
    ]
)
summary

## Notes
- `ce` does not use rollout weighting and serves as a supervised baseline.

- `winner_only` and `raft` can have zero-gradient batches if no sampled rollout is correct; this effect is captured by `train_has_winner`.
- Increase `N` to reduce no-winner batches but at higher rollout compute cost.
- If runtime is high in Colab CPU, keep `FULL_COMPARE=False` and maybe reduce `N_VALUES` to `[2, 8]`.